In [3]:
def dfs(grafo, inicio):
    visitados = set()
    orden = []

    def _dfs(nodo):
        visitados.add(nodo)
        orden.append(nodo)
        for vecino in grafo[nodo]:
            if vecino not in visitados:
                _dfs(vecino)
    _dfs(inicio)
    return orden

def dfs_iterativo(grafo, inicio):
    visitados = set()
    pila = [inicio]
    orden = []

    while pila:
        nodo  = pila.pop()
        if nodo not in visitados:
            visitados.add(nodo)
            orden.append(nodo)
            for vecino in reversed(grafo[nodo]): # Reversed for correct DFS traversal order when using a stack
                if vecino not in visitados:
                    pila.append(vecino)

    return orden


grafo = {
    'A': ['B','C'],
    'B':['D','E'],
    'C':['F'],
    'D':[],
    'E':['F'],
    'F':[]
}



print(dfs(grafo, 'A'))
print(dfs_iterativo(grafo, 'A'))

['A', 'B', 'D', 'E', 'F', 'C']
['A', 'B', 'D', 'E', 'F', 'C']


In [6]:
from collections import deque

def bfs(grafo, inicio):
    visitados = set()
    cola = deque([inicio])
    visitados.add(inicio) # Mark the starting node as visited when it's first added to the queue
    orden = []

    while cola:
        nodo = cola.popleft()
        orden.append(nodo) # Add node to order when it's dequeued

        for vecino in grafo[nodo]:
            if vecino not in visitados:
                visitados.add(vecino) # Mark neighbor as visited when adding to the queue
                cola.append(vecino)

    return orden


grafo = {
    'A': ['B','C'],
    'B':['D','E'],
    'C':['F'],
    'D':[],
    'E':['F'],
    'F':[]
}

print(bfs(grafo, 'A'))

['A', 'B', 'C', 'D', 'E', 'F']


In [9]:
import heapq

def dijkstra(grafo, inicio):
    dist = {nodo: float('inf') for nodo in grafo}
    dist[inicio] = 0
    prev = {nodo: None for nodo in grafo}

    heap = [(0, inicio)]

    while heap:
        d, u = heapq.heappop(heap)

        if d > dist[u]:
             continue


        for v, peso in grafo[u]:
            nueva_dist = dist[u] + peso
            if nueva_dist < dist[v]:
                dist[v] = nueva_dist
                prev[v] = u
                heapq.heappush(heap, (nueva_dist, v))

    return dist, prev


def reconstruir_camino(prev, inicio, fin):
    camino = []
    nodo = fin
    while nodo is not None and nodo in prev: # Added 'and nodo in prev' check for robustness
        camino.append(nodo)
        nodo = prev[nodo]
    camino.reverse()
    # Check if the path actually starts from 'inicio' and ends at 'fin'
    # The path is valid if 'inicio' is the first element after reverse and 'fin' was found.
    if not camino or camino[0] != inicio or (fin not in prev and inicio != fin): # Handle case where fin is unreachable or not in graph
        return []
    return camino


grafo = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 3)],
    'C': [('F', 5)],
    'D': [], # Added for completeness of nodes
    'E': [('F', 1)], # Example to connect E to F
    'F': [] # Added for completeness of nodes
}


dist, prev = dijkstra(grafo, 'A')
print(dist)

print(reconstruir_camino(prev, 'A', 'F'))

{'A': 0, 'B': 1, 'C': 4, 'D': 3, 'E': 4, 'F': 5}
['A', 'B', 'E', 'F']


In [11]:
def bellman_ford(grafo, inicio):
    nodos = set()
    # Correct tuple unpacking: u, v, _ (for the weight, which we don't need for node set)
    for u, v, _ in grafo:
        nodos.add(u)
        nodos.add(v)
    nodos.add(inicio)

    dist = {n: float('inf') for n in nodos}
    prev = {n: None for n in nodos}
    dist[inicio] = 0

    V = len(nodos)

    for i in range(V - 1):
        actualizado = False
        # Correct tuple unpacking: u, v, peso
        for u, v, peso in grafo:
            if dist[u] != float('inf') and dist[u] + peso < dist[v]:
                dist[v] = dist[u] + peso
                prev[v] = u
                actualizado = True
        if not actualizado:
            break

    # Negative cycle detection loop should be inside the function
    for u, v, peso in grafo:
        if dist[u] != float('inf') and dist[u] + peso < dist[v]:
            raise ValueError(f"Ciclo negativo detectado en arista {u} -> {v}")

    # Return statement should be inside the function
    return dist, prev


def reconstruir_camino(prev, inicio, fin):
    camino, nodo = [], fin
    while nodo is not None:
        camino.append(nodo)
        nodo = prev[nodo]
    camino.reverse()
    return camino if camino[0] == inicio else []


aristas = [
    ('A', 'B', 1),
    ('A', 'C', 4),
    ('B', 'D', 2),
    ('B', 'E', 3),
    ('C', 'F', 5),
]


dist, prev =  bellman_ford(aristas, 'A')
print(dist)

print(reconstruir_camino(prev, 'A', 'F'))

{'D': 3, 'A': 0, 'C': 4, 'B': 1, 'E': 4, 'F': 9}
['A', 'C', 'F']


In [13]:
class UnionFind:
    def __init__(self, nodos):
        self.padre = {n: n for n in nodos}
        self.rango = {n: 0 for n in nodos}


    def find(self, x):
        # Path compression
        if self.padre[x] != x:
            self.padre[x] = self.find(self.padre[x])
        return self.padre[x] # Corrected: should return the root of the set x belongs to

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y) # Corrected: changed '==' to '='
        if rx == ry: return False

        # Union by rank
        if self.rango[rx] < self.rango[ry]:
            rx, ry = ry, rx

        self.padre[ry] = rx
        if self.rango[rx] == self.rango[ry]:
            self.rango[rx] += 1 # Corrected: typo 'rando' to 'rango'
        return True


def kruskal(nodos, aristas):
    uf = UnionFind(nodos)
    mst = []
    # Sort edges by weight
    for peso, u, v in sorted(aristas):
        if uf.union(u, v):
            mst.append((u, v, peso)) # Corrected: append a tuple (u,v,peso) and removed extra colon
    return mst


import heapq

def prim(grafo, inicio):
    visitados = {inicio}
    # heap stores (peso, u, v)
    heap = [(peso, inicio, v) for v, peso in grafo[inicio]] # Corrected: 'iin' to 'in'
    heapq.heapify(heap)
    mst = []

    while heap and len(visitados) < len(grafo):
        peso, u, v = heapq.heappop(heap)

        if v in visitados:
            continue

        visitados.add(v)
        mst.append((u, v, peso)) # Storing the edge as (u, v, peso)

        # Add all edges from the newly visited node 'v' to the heap
        for vecino, p in grafo[v]:
            if vecino not in visitados:
                heapq.heappush(heap, (p, v, vecino))

    return mst


nodos   = ['A','B','C','D','E','F']
aristas = [
    (4,'A','B'),(2,'A','C'),(1,'C','B'),(5,'B','D'),
    (3,'C','E'),(1,'B','E'),(4,'E','D'),(2,'D','F'),(5,'E','F'),
]

grafo = {
    'A':[('B',4),('C',2)],         'B':[('A',4),('C',1),('D',5),('E',1)],
    'C':[('A',2),('B',1),('E',3)], 'D':[('B',5),('E',4),('F',2)],
    'E':[('C',3),('B',1),('D',4),('F',5)], 'F':[('D',2),('E',5)],
}

print("Kruskal:", kruskal(nodos, aristas))
print("Prim:   ", prim(grafo, 'A'))

Kruskal: [('B', 'E', 1), ('C', 'B', 1), ('A', 'C', 2), ('D', 'F', 2), ('E', 'D', 4)]
Prim:    [('A', 'C', 2), ('C', 'B', 1), ('B', 'E', 1), ('E', 'D', 4), ('D', 'F', 2)]
